In [0]:
df_emp = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/employees")
df_dept = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/departments")
df_hires = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/new_hires")
df_bands = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/salary_bands") 
df_employees_clean = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/silver/employees_clean") 

display(df_emp)
display(df_dept)
display(df_hires)
display(df_bands)
display(df_employees_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5
4,David,HR,61000,2022-01-10,M,27,true,6100,4
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3


dept,location,manager,budget
Engineering,New York,Sara,500000
Marketing,Chicago,Tom,300000
HR,Austin,Asha,200000
Finance,Dallas,Leo,400000


id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3


dept,gender,band_label
Engineering,F,Band-A
Engineering,M,Band-A
Marketing,F,Band-B
Marketing,M,Band-B
HR,F,Band-C
HR,M,Band-C
Finance,F,Band-B
Finance,M,Band-B


id,name,department,salary,join_date,age,is_active,bonus,rating,experience_level,days_employed,salary_band
1,Alice,Engineering,95000,2021-03-15,29,true,9500,4,Mid,1874,High
2,Bob,Marketing,72000,2019-07-01,35,true,7200,3,Mid,2497,Mid
4,David,HR,61000,2022-01-10,27,true,6100,4,Junior,1573,Low
5,Eve,Marketing,79000,2018-05-25,38,true,7900,4,Senior,2899,Mid
7,Grace,HR,58000,2023-02-14,24,true,5800,3,Junior,1173,Low


In [0]:
df_all = df_emp.unionByName(df_hires)
display(df_all)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5
4,David,HR,61000,2022-01-10,M,27,true,6100,4
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3


In [0]:
silver_path = "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/silver/"

In [0]:
from pyspark.sql.functions import col

df_emp_clean = spark.read.parquet(
    "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/raw/new_hires.parquet"
).withColumn("join_date", col("join_date").cast("Date"))

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3


In [0]:
from pyspark.sql.functions import year, when, col

df_emp_clean = df_emp_clean.withColumn("join_year", year(col("join_date")))

df_emp_clean = df_emp_clean.withColumn(
    "experience_level", 
    when(col("join_year") < 2019, "Senior")
    .when((col("join_year") >= 2019) & (col("join_year") <= 2021), "Mid")
    .otherwise("Junior")
).drop("join_year")  

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,experience_level
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4,Junior
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5,Junior
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3,Junior


In [0]:
from pyspark.sql.functions import col, datediff, lit, to_date, current_timestamp

df_emp_clean = df_emp_clean.withColumn("days_employed", datediff(to_date(current_timestamp(), "yyyy-MM-dd"), col("join_date")))

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,experience_level,days_employed
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4,Junior,848
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5,Junior,1005
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3,Junior,783


In [0]:
from pyspark.sql.functions import year, when, col, datediff, lit, to_date, current_timestamp

df_emp_clean = df_emp_clean.withColumn(
    "salary_band",
    when(col("salary") >= 90000, "High")
    .when(col("salary") >= 70000, "Mid")
    .otherwise("Low")
)

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,experience_level,days_employed,salary_band
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4,Junior,848,Mid
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5,Junior,1005,High
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3,Junior,783,Low


In [0]:
df_emp_clean = df_emp_clean.withColumnRenamed("dept", "department")
df_emp_clean = df_emp_clean.drop("gender")

display(df_emp_clean)

id,name,department,salary,join_date,age,is_active,bonus,rating,experience_level,days_employed,salary_band
8,Heidi,Finance,85000,2024-01-05,28,true,8500,4,Junior,848,Mid
9,Ivan,Engineering,91000,2023-08-01,30,true,9100,5,Junior,1005,High
10,Judy,Marketing,67000,2024-03-10,26,true,6700,3,Junior,783,Low


In [0]:
df_all_clean = df_employees_clean.unionByName(df_emp_clean)
display(df_all_clean)

id,name,department,salary,join_date,age,is_active,bonus,rating,experience_level,days_employed,salary_band
1,Alice,Engineering,95000,2021-03-15,29,true,9500,4,Mid,1874,High
2,Bob,Marketing,72000,2019-07-01,35,true,7200,3,Mid,2497,Mid
4,David,HR,61000,2022-01-10,27,true,6100,4,Junior,1573,Low
5,Eve,Marketing,79000,2018-05-25,38,true,7900,4,Senior,2899,Mid
7,Grace,HR,58000,2023-02-14,24,true,5800,3,Junior,1173,Low
8,Heidi,Finance,85000,2024-01-05,28,true,8500,4,Junior,848,Mid
9,Ivan,Engineering,91000,2023-08-01,30,true,9100,5,Junior,1005,High
10,Judy,Marketing,67000,2024-03-10,26,true,6700,3,Junior,783,Low


In [0]:
df_emp_clean.write.mode("overwrite").parquet(silver_path + "new_hires_clean")
df_all_clean.write.mode("overwrite").parquet(silver_path+ "employees_all_clean")

In [0]:
df_final = df_all.join(df_dept, on='dept', how='inner').select(
    df_all["*"],
    df_dept["location"],
    df_dept["manager"],
    df_dept["budget"]
)

display(df_final)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,location,manager,budget
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4,New York,Sara,500000
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3,Chicago,Tom,300000
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5,New York,Sara,500000
4,David,HR,61000,2022-01-10,M,27,true,6100,4,Austin,Asha,200000
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4,Chicago,Tom,300000
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5,New York,Sara,500000
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3,Austin,Asha,200000
8,Heidi,Finance,85000,2024-01-05,F,28,true,8500,4,Dallas,Leo,400000
9,Ivan,Engineering,91000,2023-08-01,M,30,true,9100,5,New York,Sara,500000
10,Judy,Marketing,67000,2024-03-10,F,26,true,6700,3,Chicago,Tom,300000


In [0]:
gold_path = "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/silver/"

In [0]:
df_final = df_all_clean.join(df_dept, df_all_clean.department == df_dept.dept, how='inner').select(
    df_all_clean["id"],
    df_all_clean["name"],
    df_all_clean["department"],
    df_dept["location"],
    df_dept["manager"],
    df_all_clean["salary"],
    df_all_clean["salary_band"],
    df_all_clean["bonus"],
    df_all_clean["experience_level"],
    df_all_clean["days_employed"],
    df_all_clean["rating"]
)

display(df_final)

id,name,department,location,manager,salary,salary_band,bonus,experience_level,days_employed,rating
1,Alice,Engineering,New York,Sara,95000,High,9500,Mid,1874,4
2,Bob,Marketing,Chicago,Tom,72000,Mid,7200,Mid,2497,3
4,David,HR,Austin,Asha,61000,Low,6100,Junior,1573,4
5,Eve,Marketing,Chicago,Tom,79000,Mid,7900,Senior,2899,4
7,Grace,HR,Austin,Asha,58000,Low,5800,Junior,1173,3
8,Heidi,Finance,Dallas,Leo,85000,Mid,8500,Junior,848,4
9,Ivan,Engineering,New York,Sara,91000,High,9100,Junior,1005,5
10,Judy,Marketing,Chicago,Tom,67000,Low,6700,Junior,783,3
